# 05 SHAP解释（任务书：SHAP、分区主控因子变化）图片为点图

In [ ]:
import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
SHAP_DIR = ensure_dir(OUT/'shap')
df = pd.read_parquet(OUT/'bcf_final.parquet')
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.explain_shap import compute_shap_for_pipeline, save_shap_summary
from src.utils import safe_col
import numpy as np

bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
cv = get_group_kfold(cfg)

preferred = [m for m in ['xgb','rf'] if m in cfg['modeling']['models']]
if not preferred:
    raise RuntimeError("请在config里至少包含 xgb 或 rf 以高效做SHAP")
m = preferred[0]

group_keys = ['crop','ph_bin'] if 'crop' in df.columns else ['ph_bin']
summary_rows = []
for key, sub in df.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)
    gsub = make_spatial_groups(sub, cfg)
    res = fit_oof_with_spatialcv(sub, cfg, gsub, m, cv)

    X = sub.drop(columns=[bcf_col], errors='ignore')
    sv, feat_names = compute_shap_for_pipeline(res.best_estimator, X, max_rows=int(cfg['shap']['sample_rows']), random_state=cfg['project']['seed'])

    key_str = "_".join(map(str, key if isinstance(key, tuple) else (key,)))
    save_shap_summary(sv, feat_names, str(SHAP_DIR/f"shap_summary_{key_str}_{m}.png"))

    mean_abs = np.mean(np.abs(sv), axis=0)
    topk = np.argsort(mean_abs)[::-1][:int(cfg['shap']['top_k'])]
    for i in topk:
        summary_rows.append({"group": str(key), "model_for_shap": m, "feature": feat_names[i], "mean_abs_shap": float(mean_abs[i])})

rank = pd.DataFrame(summary_rows)
rank.to_excel(OUT/'shap_rank_compare.xlsx', index=False)
rank.head(20)



## 论文风格 SHAP 重要性图（条形 + 蜂群）

本节输出类似论文常见的 **Mean|SHAP| 条形 + SHAP 蜂群散点** 组合图（更直观）。
输出目录：`outputs/shap/`。


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.utils import load_config, ensure_dir
from src.shap_plots import shap_importance_combo

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
shap_dir = ensure_dir(OUT / 'shap')
cache_dir = OUT / 'shap_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

def load_cache_items():
    items = []
    for npz in sorted(cache_dir.glob('shap_*.npz')):
        data = np.load(npz, allow_pickle=True)
        items.append((npz.stem.replace('shap_',''), data['shap_values'], data['X'], data['feature_names'].tolist(), str(data['title'])))
    return items

items = load_cache_items()
print('cache items:', len(items))
for key, sv, X, names, title in items:
    out_png = shap_dir / f'shap_combo_{key}.png'
    shap_importance_combo(sv, X, names, out_png, title=title)
    print('saved', out_png)

if len(items)==0:
    print('未发现 outputs/shap_cache/shap_*.npz。请在上游 SHAP 计算循环里保存缓存（见下面提示）。')


### 如何在现有 SHAP 计算循环里保存缓存（必须一次）

在你已有的 SHAP 计算代码里（每个 crop×ph_bin×model 计算完以后），加：

```python
cache_dir = OUT / 'shap_cache'
cache_dir.mkdir(parents=True, exist_ok=True)
np.savez(
    cache_dir / f'shap_{crop}_{ph}_{model}.npz',
    shap_values=sv,  # 形状 (n_samples, n_features)
    X=X_used.values, # 对应的特征矩阵
    feature_names=np.array(feature_names, dtype=object),
    title=f"({crop}, {ph}) | {model} SHAP"
)
```
